# Computational Assignment #5: Logistic Regression Computations
#### MSDS 410

---

In this assignment you will be calculating summary statistics associated with logistic regression, fitting logistic regression models and interpreting the results. 

Show all decimals to three places, X.xxx. 

Any computations that involve "the log function", denoted by log(x) ***mean the natural log function (which will show as ln() on a calculator)***. 

---

## Q1

For the 2x2 table, calculate the odds and the probabilities of texting while driving for males and females. Then compute the odds ratio of texting while driving that compares males to females. (5 points)

| Texting While Driving | Male | Female |
| --- | --- | --- |
| YES | 30 | 34 |
| NO | 10 | 6 |

Using Bayes' Theorem of general probability, we can calculate approximately the probability that a person texts while driving. To set this up, we can assume the following: 
- P(YES | Male) = 0.75
- P(YES | Female) = 0.85
- P(NO | Male) = 0.25
- P(NO | Female) = 0.15

$$OR = \frac{Odds(Male)}{Odds(Female)} = \frac{P(Yes|M)/P(No|M)}{P(Yes|F)/P(No|F)}$$
$$OR = \frac{P(Yes|M) \times P(No|F)}{P(No|M) \times P(Yes|F)}$$
$$OR = \frac{0.75 \times 0.15}{0.85 \times 0.25} = 0.529$$

In [3]:
(0.75*0.15)/(0.85*0.25)

0.5294117647058824

**Interpretation**: the odds of texting while driving for males are about 0.529 times the odds for females. Males have lower odds of texting while driving than females in this sample (~47% lower odds)

In [5]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option('display.float_format', lambda x: f'{x:.3f}')

df = pd.read_csv('RELIGION.csv')
df.shape

(626, 10)

--- 
## Q2

Download the data file `RELIGION.CSV` and import into Python. Use Python and your EDA skills to gain a basic understanding of this dataset. `RELSCHOL` indicates if a survey respondent attends a religiously affiliated private secondary school (1 = Yes) or not (0 = No). Use this dataset to address the following questions (10 points): 

Compute the overall odds and probability of attending a religious school. 

Cross-tabulate `RELSCHOL` with `RACE` (coded: 0=non-white, 1=white). What are the probabilities that non-white students and white students attend religious schools?

What are the odds that white students and non-white students attend religious schools?

What is the odds ratio that compares white and non-white students?

In [7]:
df.head()

,ID,SEX,AGE,EDUC,INCOME,RELSCHOL,MARRIED,ATTEND,AGESQ,RACE
0,2,1,30,6,11,0,1,6,900,1
1,3,1,32,6,6,0,1,5,1024,1
2,4,1,51,2,11,0,1,2,2601,1
3,5,1,18,2,3,0,0,6,324,1
4,6,1,37,5,6,0,1,6,1369,1


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 626 entries, 0 to 625
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   ID        626 non-null    int64
 1   SEX       626 non-null    int64
 2   AGE       626 non-null    str  
 3   EDUC      626 non-null    str  
 4   INCOME    626 non-null    str  
 5   RELSCHOL  626 non-null    int64
 6   MARRIED   626 non-null    int64
 7   ATTEND    626 non-null    int64
 8   AGESQ     626 non-null    str  
 9   RACE      626 non-null    int64
dtypes: int64(6), str(4)
memory usage: 53.8 KB


### 1. EDA / Cleaning

A few cells in the raw CSV are blank, which makes pandas read those columns as text instead of numbers. Converting them to numeric turns the blanks into `NaN`, and we then drop any row that's missing a value we need. 

In [10]:
# Convert to numeric (blanks become NaN)
numeric_cols = ['AGE', 'EDUC', 'INCOME', 'AGESQ']
for c in numeric_cols: 
    df[c] = pd.to_numeric(df[c], errors='coerce')

print("Missing values per column:")
print(df.isna().sum())

df = df.dropna().reset_index(drop=True)
print(f"\nRows remaining after dropping missing values: {len(df)}")

Missing values per column:
ID           0
SEX          0
AGE          3
EDUC         2
INCOME      36
RELSCHOL     0
MARRIED      0
ATTEND       0
AGESQ        3
RACE         0
dtype: int64

Rows remaining after dropping missing values: 587


### 2. Overall odds and probability of attending a religious school

- **Probability** = P(RELSCHOL = 1) = (# yes) / (total n)
- **Odds** = P(yes) / P(no) = (# yes) / (# no)

In [12]:
n_total = len(df)
n_yes = (df['RELSCHOL'] == 1).sum()
n_no = (df['RELSCHOL'] == 0).sum()

prob_relschol = n_yes / n_total
odds_relschol = n_yes / n_no

print(f"n = {n_total}, yes = {n_yes}, no = {n_no}")
print(f"Overall probability of attending a religious school: {prob_relschol:.3f}")
print(f"Overall odds of attending a religious schoo: {odds_relschol:.3f}")

n = 587, yes = 76, no = 511
Overall probability of attending a religious school: 0.129
Overall odds of attending a religious schoo: 0.149


### Cross-tabulate RELSCHOL with RACE

- P(RELSCHOL = 1 | RACE = white) and P(RELSCHOL = 1 | RACE = non-white)
- Odds of RELSCHOL = 1 for white and non-white students
- Odds ratio of comparing white to non-white

In [14]:
ct = pd.crosstab(df['RACE'], df['RELSCHOL'], rownames=['RACE (0=non-white, 1=white)'], colnames=['RELSCHOL'])
ct

RELSCHOL,0,1
"RACE (0=non-white, 1=white)",,
0,74,25
1,437,51


In [15]:
# Counts
white_yes = ct.loc[1, 1]
white_no = ct.loc[1, 0]
nonwhite_yes = ct.loc[0, 1]
nonwhite_no = ct.loc[0, 0]

# Probabilities
p_relschol_white = white_yes / (white_yes + white_no)
p_relschol_nonwhite = nonwhite_yes / (nonwhite_yes + nonwhite_no)

print(f"P(RELSCHOL = 1 | white) = {p_relschol_white:.3f}")
print(f"P(RELSCHOL = 1 | non-white) = {p_relschol_nonwhite:.3f}")

P(RELSCHOL = 1 | white) = 0.105
P(RELSCHOL = 1 | non-white) = 0.253


In [16]:
# Odds
odds_white = white_yes / white_no
odds_nonwhite = nonwhite_yes / nonwhite_no

print(f"Odds (RELSCHOL=1 | white) = {odds_white:.3f}")
print(f"Odds (RELSCHOL=1 | non-white) = {odds_nonwhite:.3f}")

Odds (RELSCHOL=1 | white) = 0.117
Odds (RELSCHOL=1 | non-white) = 0.338


In [17]:
# Odds ratio: white vs non-white
odds_ratio_white_vs_nonwhite = odds_white / odds_nonwhite

# ad/bc cross-product shortcut
odds_ratio_crossproduct = (white_yes * nonwhite_no) / (white_no * nonwhite_yes)

print(f"Odds ratio (white vs non-white): {odds_ratio_white_vs_nonwhite:.3f}")
print(f"Cross-product check: {odds_ratio_crossproduct:.3f}")

Odds ratio (white vs non-white): 0.345
Cross-product check: 0.345


**Interpretation**: The probability of a white student attending a religious school is roughly 10.5% while the probability of a non-white student is roughly 25.3%. The odds of a white student attending a religious school are about 0.345 times the odds of a non-white student. White students have lower odds of attending a religious school compared to non-white students (~65.5%).

### 3. Model 1 - Logistic regression: RELSCHOL ~ RACE

$$logit(p) = log(\frac{p}{1-p}) = \beta_0 + \beta_1 \times RACE$$

In [19]:
model1 = smf.logit('RELSCHOL ~ RACE', data=df).fit()
print(model1.summary())

Optimization terminated successfully.
         Current function value: 0.373704
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               RELSCHOL   No. Observations:                  587
Model:                          Logit   Df Residuals:                      585
Method:                           MLE   Df Model:                            1
Date:                Mon, 10 Aug 2026   Pseudo R-squ.:                 0.03030
Time:                        19:37:22   Log-Likelihood:                -219.36
converged:                       True   LL-Null:                       -226.22
Covariance Type:            nonrobust   LLR p-value:                 0.0002134
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -1.0852      0.231     -4.691      0.000      -1.539      -0.632
RACE          -1.0629      0.

In [20]:
print(f"Model 1 AIC: {model1.aic:.3f}")
print(f"Model 1 BIC: {model1.bic:.3f}")
print()
print("Coefficients:")
print(model1.params.round(3))
print()
print("Odds ratios (exp(coef)):")
print(np.exp(model1.params).round(3))

Model 1 AIC: 442.728
Model 1 BIC: 451.478

Coefficients:
Intercept   -1.085
RACE        -1.063
dtype: float64

Odds ratios (exp(coef)):
Intercept   0.338
RACE        0.345
dtype: float64


**Interpretation**:

$$logit(p) = log(\frac{p}{1-p}) = \beta_0 + \beta_1 \times RACE = -1.085 - 1.063 \times RACE$$

- $\beta_0$ = -1.085
- $\beta_1$ = -1.063

For non-white students: 
$$(RACE=0):logit(p) = -1.085 - 1.063(0) = -1.085$$

For white students: 
$$(RACE=1):logit(p) = -1.085 - 1.063(1) = -2.148$$

$$e^{\beta_1} = e^{-1.063} = 0.345$$

This ties to our manual crossmultiply above: The odds of a white student attending a religious school are about 0.345 times the odds of a non-white student. 

### 4. Model 2 - Logistic regression: RELSCHOL ~ INCOME

#### a. Fit the model, report AIC/BIC, interpret coefficients

In [23]:
model2 = smf.logit('RELSCHOL ~ INCOME', data=df).fit()
print(model2.summary())

Optimization terminated successfully.
         Current function value: 0.375299
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               RELSCHOL   No. Observations:                  587
Model:                          Logit   Df Residuals:                      585
Method:                           MLE   Df Model:                            1
Date:                Mon, 10 Aug 2026   Pseudo R-squ.:                 0.02616
Time:                        19:37:22   Log-Likelihood:                -220.30
converged:                       True   LL-Null:                       -226.22
Covariance Type:            nonrobust   LLR p-value:                 0.0005805
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -2.8116      0.306     -9.186      0.000      -3.411      -2.212
INCOME         0.1616      0.

In [24]:
print(f"Model 2 AIC: {model2.aic:.3f}")
print(f"Model 2 BIC: {model2.bic:.3f}")
print()
print("Coefficients:")
print(model2.params.round(3))
print()
print("Odds ratio for a 1-unit increase in INCOME:", round(np.exp(model2.params['INCOME']), 3))

Model 2 AIC: 444.601
Model 2 BIC: 453.351

Coefficients:
Intercept   -2.812
INCOME       0.162
dtype: float64

Odds ratio for a 1-unit increase in INCOME: 1.175


**Interpretation**: Model 1's AIC and BIC (442 & 451, respectively) are both lower than Model 2's (444 & 453, respectively). Additionally, Model 1's Pseudo $R^2$ is higher at 0.030 compared to Model 2 at 0.026. All three metrics point to Model 1 (RACE) fitting the data slightly better than Model 2 (INCOME). The INCOME coefficient is 0.162, so $e^{0.162} = 1.175$, meaning each one-unit increase in the INCOME category is associated with a 17.5% increase in the odds of attending a religious school, holding nothing else constant since this is a single-predictor model. 

#### b. Predicted probabilities from Model 2

Compute the predicted probability for every record using the fitted logit equation, sort them, and find the smallest INCOME value at which the predicted probability first exceeds 0.50. 

In [26]:
df['pred_prob_model2'] = model2.predict(df)

# Sorted view of predicted probabilities alongside INCOME
sorted_m2 = df[['INCOME', 'pred_prob_model2']].sort_values('pred_prob_model2').reset_index(drop=True)
sorted_m2

,INCOME,pred_prob_model2
0,1.000,0.066
1,1.000,0.066
2,1.000,0.066
3,1.000,0.066
4,1.000,0.066
...,...,...
582,12.000,0.295
583,12.000,0.295
584,12.000,0.295
585,12.000,0.295


In [27]:
# Find the INCOME threshold where predicted probability first exceeds 0.50
threshold_row = sorted_m2[sorted_m2['pred_prob_model2'] > 0.50].head(1)
print("First record where predicted probability exceeds 0.50:")
print(threshold_row)

# Solve algebraically; logit(p)=0 when p=0.50, so INCOME = -b0/b1
b0, b1 = model2.params['Intercept'], model2.params['INCOME']
income_star = -b0 / b1
print(f"\nAlgebraic solution: predicted probability = 0.50 at INCOME = {income_star:.3f}")

First record where predicted probability exceeds 0.50:
Empty DataFrame
Columns: [INCOME, pred_prob_model2]
Index: []

Algebraic solution: predicted probability = 0.50 at INCOME = 17.402


### 5. Model 3 - Logistic regression: RELSCHOL ~ ATTEND

#### a. Fit the model, report AIC/BIC, interpret coefficients

In [29]:
model3 = smf.logit('RELSCHOL ~ ATTEND', data=df).fit()
print(model3.summary())

Optimization terminated successfully.
         Current function value: 0.381268
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               RELSCHOL   No. Observations:                  587
Model:                          Logit   Df Residuals:                      585
Method:                           MLE   Df Model:                            1
Date:                Mon, 10 Aug 2026   Pseudo R-squ.:                 0.01067
Time:                        19:37:22   Log-Likelihood:                -223.80
converged:                       True   LL-Null:                       -226.22
Covariance Type:            nonrobust   LLR p-value:                   0.02799
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -3.1079      0.598     -5.195      0.000      -4.281      -1.935
ATTEND         0.2586      0.

In [30]:
print(f"Model 3 AIC: {model3.aic:.3f}")
print(f"Model 3 BIC: {model3.bic:.3f}")
print()
print("Coefficients:")
print(model3.params.round(3))
print()
print("Odds ratio for a 1-unit increase in ATTEND:", round(np.exp(model3.params['ATTEND']), 3))

Model 3 AIC: 451.609
Model 3 BIC: 460.359

Coefficients:
Intercept   -3.108
ATTEND       0.259
dtype: float64

Odds ratio for a 1-unit increase in ATTEND: 1.295


**Interpretation**: Model 3's AIC (451) and BIC (460) are both higher than Model 1's and Model 2's. Model 3's Pseudo $R^2$ is also lower than both at 0.011. With all three metrics worse than our first two models, Model 3 (ATTEND) has the worst fit of all of our models so far. The ATTEND coefficient is 0.259, which corresponds to an odds ratio of $e^{0.259} = 1.295$, meaning a one-unit increase in the ATTEND category is associated with a 29.5% increase in the odds of attending a religious schoo, holding nothing else constant.

#### b. Predicted probabilities from Model 3 - find the ATTEND threshold

In [32]:
df['pred_prob_model3'] = model3.predict(df)

sorted_m3 = df[['ATTEND', 'pred_prob_model3']].sort_values('pred_prob_model3').reset_index(drop=True)
sorted_m3

,ATTEND,pred_prob_model3
0,1,0.055
1,1,0.055
2,1,0.055
3,1,0.055
4,1,0.055
...,...,...
582,6,0.174
583,6,0.174
584,6,0.174
585,6,0.174


In [33]:
threshold_row3 = sorted_m3[sorted_m3['pred_prob_model3'] > 0.50].head(1)
print("First record where predicted probability exceeds 0.50:")
print(threshold_row3)

b0, b1 = model3.params['Intercept'], model3.params['ATTEND']
attend_star = -b0 / b1
print(f"\nAlgebraic solution: predicted probability = 0.50 at ATTEND = {attend_star:.3f}")

First record where predicted probability exceeds 0.50:
Empty DataFrame
Columns: [ATTEND, pred_prob_model3]
Index: []

Algebraic solution: predicted probability = 0.50 at ATTEND = 12.019


### 6. Model 4 - Logistic regression: RELSCHOL ~ RACE + INCOME + ATTEND

#### a. Fit the model, report AIC/BIC, interpret coefficients

In [35]:
model4 = smf.logit('RELSCHOL ~ RACE + INCOME + ATTEND', data=df).fit()
print(model4.summary())

Optimization terminated successfully.
         Current function value: 0.354566
         Iterations 7
                           Logit Regression Results                           
Dep. Variable:               RELSCHOL   No. Observations:                  587
Model:                          Logit   Df Residuals:                      583
Method:                           MLE   Df Model:                            3
Date:                Mon, 10 Aug 2026   Pseudo R-squ.:                 0.07996
Time:                        19:37:22   Log-Likelihood:                -208.13
converged:                       True   LL-Null:                       -226.22
Covariance Type:            nonrobust   LLR p-value:                 6.868e-08
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -3.5754      0.718     -4.983      0.000      -4.982      -2.169
RACE          -1.2825      0.

In [36]:
print(f"Model 4 AIC: {model4.aic:.3f}")
print(f"Model 4 BIC: {model4.bic:.3f}")
print()
print("Coefficients:")
print(model4.params.round(3))
print()
print("Odds ratios (exp(coef)) for each predictor:")
print(np.exp(model4.params).round(3))

Model 4 AIC: 424.260
Model 4 BIC: 441.760

Coefficients:
Intercept   -3.575
RACE        -1.283
INCOME       0.200
ATTEND       0.331
dtype: float64

Odds ratios (exp(coef)) for each predictor:
Intercept   0.028
RACE        0.277
INCOME      1.221
ATTEND      1.392
dtype: float64


**Interpretation**: Model 4 is our best fitted model so far with the lowest AIC (424), lowest BIC (441), and highest Pseudo $R^2$ (0.080). Model 4's lower AIC/BIC is the more meaningful result, since those metrics are working against it as it adds more predictors. This is evidence that the added variables are all pulling their weight and not just padding the model (increasing Pseudo $R^2$). Interestingly, RACE's effect got stronger in Model 4 (OR dropped from 0.345 to 0.277) once INCOME and ATTEND were controlled for. 

| Predictor | $\beta$ (log-odds) | $e^{\beta}$ (odds ratio) | Interpretation | 
| --- | --- | --- | --- |
| RACE | -1.283 | 0.277 | Holding INCOME and ATTEND constant, white students have odds of attending a religious school that are about 0.277 times (72.3% lower) those of non-white students | 
| INCOME | 0.200 | 1.221 | Holding RACE and ATTEND constant, each one-unit increase in the INCOME category is associated with a 22.1% increase in the odds of attending a religious school. |
| ATTEND | 0.331 | 1.392 | Holding RACE and INCOME constant, each one-unit increase in the ATTEND category is associated with a 39.2% increase in the odds of attending a religious school. |

#### b. Predicted odds for ATTEND = 5, INCOME = 4 - white vs. non-white

In [38]:
b0 = model4.params['Intercept']
b_race = model4.params['RACE']
b_income = model4.params['INCOME']
b_attend = model4.params['ATTEND']

attend_val = 5
income_val = 4

logit_white = b0 + b_race * 1 + b_income * income_val + b_attend * attend_val
logit_nonwhite = b0 + b_race * 0 + b_income * income_val + b_attend * attend_val

odds_white_m4 = np.exp(logit_white)
odds_nonwhite_m4 = np.exp(logit_nonwhite)

prob_white_m4 = odds_white_m4 / (1 + odds_white_m4)
prob_nonwhite_m4 = odds_nonwhite_m4 / (1 + odds_nonwhite_m4)

print(f"White, ATTEND=5, INCOME=4: logit = {logit_white:.3f}, odds = {odds_white_m4:.3f}, prob = {prob_white_m4:.3f}")
print(f"Non-white, ATTEND=5, INCOME=4: logit = {logit_nonwhite:.3f}, odds = {odds_nonwhite_m4:.3f}, prob = {prob_nonwhite_m4:.3f}")
print(f"\nOdds ratio (white vs non-white) at these covariate values: {odds_white_m4/odds_nonwhite_m4:.3f}")

White, ATTEND=5, INCOME=4: logit = -2.404, odds = 0.090, prob = 0.083
Non-white, ATTEND=5, INCOME=4: logit = -1.122, odds = 0.326, prob = 0.246

Odds ratio (white vs non-white) at these covariate values: 0.277


### 7. Classification tables for Models 1, 2, and 3

Classification rule: **predicted probability < 0.50 -> predict 0, otherwise predict 1**. For each model, cross-tabulate actual RELSCHOL against the predicted class, then compute the correct classification rate = (correctly classified). 

In [63]:
def classify_and_report(model, model_name, df):
    pred_prob = model.predict(df)
    pred_class = (pred_prob >= 0.50).astype(int)
    ct = pd.crosstab(df['RELSCHOL'], pred_class, rownames=['Actual'], colnames=['Predicted'])
    # make sure both 0/1 columns exist even if a class was never predicted
    for c in [0, 1]: 
        if c not in ct.columns:
            ct[c] = 0
    ct = ct[[0, 1]]
    correct = np.trace(ct.values)
    total = ct.values.sum()
    rate = correct / total
    print(f"--- {model_name} ---")
    print(ct)
    print(f"Correct classification rate: {rate:.3f}\n")
    return ct, rate

ct1, rate1 = classify_and_report(model1, "Model 1 (RACE)", df)
ct2, rate2 = classify_and_report(model2, "Model 2 (INCOME)", df)
ct3, rate3 = classify_and_report(model3, "Model 3 (ATTEND)", df)

--- Model 1 (RACE) ---
Predicted    0  1
Actual           
0          511  0
1           76  0
Correct classification rate: 0.871

--- Model 2 (INCOME) ---
Predicted    0  1
Actual           
0          511  0
1           76  0
Correct classification rate: 0.871

--- Model 3 (ATTEND) ---
Predicted    0  1
Actual           
0          511  0
1           76  0
Correct classification rate: 0.871



In [77]:
summary = pd.DataFrame({
    'Model': ['Model 1 (RACE)', 'Model 2 (INCOME)', 'Model 3 (ATTEND)'], 
    'AIC': [model1.aic, model2.aic, model3.aic],
    'BIC': [model1.bic, model2.bic, model3.bic], 
    'Correct Classification Rate': [rate1, rate2, rate3]
})
summary.round(3)

,Model,AIC,BIC,Correct Classification Rate
0,Model 1 (RACE),442.728,451.478,0.871
1,Model 2 (INCOME),444.601,453.351,0.871
2,Model 3 (ATTEND),451.609,460.359,0.871


In [75]:
ct4, rate4 = classify_and_report(model4, "Model 4 (COMBINED)", df)

model4_summary = pd.DataFrame({
    'Model 4': ['Combined'],
    'AIC': [model4.aic], 
    'BIC': [model4.bic],
    'Correct Classification Rate': [rate4]
})
model4_summary.round(3)

--- Model 4 (COMBINED) ---
Predicted    0  1
Actual           
0          507  4
1           75  1
Correct classification rate: 0.865



,Model 4,AIC,BIC,Correct Classification Rate
0,Combined,424.260,441.760,0.865


**Interpretation**: Every single model predicted **class 0 for every one of the 587 records**, with zero predictions of class 1. This is because RELSCHOL = 1 is rare in this data (only 76/587 = 12.9% of students attend a religious school). None of these three single-predictor models generate a predicted probability high enough to cross the 0.50 threshold for any record because the predictors aren't strong enough on their own to push any case's probability past 50%. So every model ends up predicting the majority class (0) across the board. The "correct classification rate" of 0.871 is just the base rate: 511/587 = 0.871 (i.e., the accuracy we'd get by guessing "no" every time, regardless of whic predictor was in the model). 

This is a pitfall with imbalanced data. A high classification rate doesn't mean the model is doing meaningful predicted work. 

### 8. Conclusion

On their own, neither race/ethnicity, income, nor religious service attendance are good indicators for attending a religious school. Models 1-3 individually only reached Pseudo $R^2$ of 0.030, 0.026, and 0.011 respectively. On the other hand, Model 4 combined all variables and reached Pseudo $R^2$ of 0.080 plus had the lowest AIC (424.260) and BIC (441.760) of all four models. This is quantitative evidence that the results become more meaningful when all variables are used together. All three predictors in Model 4 have p-values well under 0.05 (RACE p<0.001, INCOME p<0.001, ATTEND p=0.011), meaning all three effects are statistically significant, not just directionally suggestive. RACE has the largest effect of the three in terms of Odds Ratio (72.3% lower odds), follwed by ATTEND (39.2% higher odds per unit), then INCOME (22.1% higher odds per unit). It is important to note that Pseudo $R^2$ of 0.080 is still fairly modest in absolute terms. This tells us that even the best model here leaves most of the variation in RELSCHOL unexplained. These three variables are meaningful but far from the whole story. 

The Model 4 classification check shows that it still struggles to correctly predict the minority class (RELSCHOL=1) given the same class-imbalance issue. Despite Model 4 having the strongest statistical fit, it correctly identifies only 1 of the 76 students who actually attend a religious school, correctly classifying the true positive rate is 1.3%. This shows that Model fit (AIC/BIC/Pseudo $R^2$) and classification accuracy under the standard 0.50 threshold are not the same thing. A well-fitting model can still perform poorly on rare-outcome classification without adjusting the decision threshold or comparing against a more informative metric like sensitivity.